# Project 07 — 06 — White's Reality Check & Hansen's SPA


## Notebook introduction

- **Concepts required:** data-snooping bias, the maximum statistic over a family, dependence-preserving bootstrap, recentring under a composite null.
- **What will be learned:** whether the *best* strategy found in the reconstructed historical search universe is genuinely superior after accounting for selection across the whole family.
- **Outputs:** the White Reality Check p-value (+ null-max percentiles), block-length sensitivity, and a comparison with the DSR evidence.
- **Not done here:** Hansen's SPA (scaffold), and any final candidate selection.

## Purpose

Given that many strategy variants were searched historically, is the best observed performance still stronger than what could plausibly arise from **data snooping**? We use White's (2000) Reality Check on the reconstructed series-backed strategy universe, reusing the canonical stationary bootstrap (`src.statistics.white_reality_check.reality_check`).

## Expected inputs

- Per-strategy performance-vs-benchmark frame for the whole family
- Bootstrap settings (resamples, block length, seed)


## Expected outputs

- Family-adjusted p-value for the best strategy
- Identity of the best strategy
- `results/reality_check.csv`
- Evidence-ledger records


## Setup & inputs


In [1]:
from pathlib import Path
import sys, glob, os, json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.statistics.white_reality_check import reality_check

WRC_SEED, WRC_N, WRC_BLOCKS, WRC_CANON = 20260820, 10000, [5, 10, 20, 30, 50], 20
print("repo root:", PROJECT_ROOT)

repo root: /Users/rawls/quant-lab


## Assemble the strategy family & benchmark

**Universe (broadest defensible, series-backed).** The ~23 reconstructed historical strategies for which aligned daily return series actually exist: **12 Project-04 base strategies** (`exp_004_project04_final/strategy_timeseries/*.csv`, `portfolio_return`) + **11 Project-05/06 US overlay variants** (`exp_006…/overlay_equity_curves.csv`, equity→returns). No returns are invented for experiments without a series (e.g. the parameter-grid rows and the metric-only cross-market sweeps are excluded).

**Gross vs net.** These are **GROSS** returns (the P04 base series and the overlay equity curves are pre-cost). Net series exist only for the 3 tournament candidates, not the whole 23, so — to avoid silently mixing scales — the Reality Check is run on **gross** returns across the whole universe, stated explicitly.

**Benchmark.** The historical search was fundamentally **strategy-vs-zero** (absolute-return long-short books, no single fixed benchmark), so the differential is `d_{k,t} = r_{k,t} − 0 = r_{k,t}` and the Reality-Check statistic is the mean return scaled by `sqrt(T)`. The best strategy is the highest-mean strategy in the family.

In [2]:
A = {}
for f in sorted(glob.glob(str(PROJECT_ROOT / "experiments/completed/exp_004_project04_final/strategy_timeseries/*.csv"))):
    A[os.path.basename(f)[:-4]] = pd.read_csv(f)["portfolio_return"].to_numpy()
A = pd.DataFrame(A)
eqB = pd.read_csv(PROJECT_ROOT / "experiments/completed/exp_006_deployment_overlay_challenge/overlay_equity_curves.csv").drop(columns=["Date"])
B = eqB.pct_change().dropna().reset_index(drop=True)
# positional alignment over the shared daily calendar (P04 dates are an integer index)
perf = pd.concat([A.iloc[1:].reset_index(drop=True).add_prefix("P04:"),
                  B.add_prefix("P06:")], axis=1)   # d_{k,t} = r_{k,t} (benchmark = 0), GROSS

C = perf.corr().values; iu = np.triu_indices_from(C, k=1)
print("strategy universe :", perf.shape[1], "strategies")
print("aligned obs (T)   :", perf.shape[0], "(gross; benchmark = 0)")
print("mean pairwise corr:", round(float(C[iu].mean()), 4), "| median:", round(float(np.median(C[iu])), 4))
print("strategies        :", list(perf.columns))

strategy universe : 23 strategies
aligned obs (T)   : 2557 (gross; benchmark = 0)
mean pairwise corr: 0.9389 | median: 0.9406
strategies        : ['P04:blend_40_60_ls20_plus_ls30', 'P04:blend_50_50_ls20_plus_ls30', 'P04:blend_60_40_ls20_plus_ls30', 'P04:blend_70_30_ls20_plus_ls30', 'P04:ls_20pct', 'P04:ls_20pct_plus_linear', 'P04:ls_20pct_plus_pow_0.7', 'P04:ls_20pct_plus_sqrt', 'P04:ls_20pct_plus_sqrt_partial', 'P04:ls_20pct_plus_sqrt_partial_normalized', 'P04:ls_30pct', 'P04:ls_30pct_plus_sqrt_partial_norm', 'P06:Deployment (incumbent, as shipped)', 'P06:Smooth DD (k5, floor0.55) [config recipe]', 'P06:DD Only (floor0.3, k5)', 'P06:Combined DD+Vol (floor0.3, k5, clip0.5-1.3)', 'P06:Floor 0.3 (m*smooth(0.3,5), clip0.5-1.3)', 'P06:Floor 0.5 (m*smooth(0.5,5), clip0.5-1.3)', 'P06:Clip 0.5-1.2 (m*smooth(0.6,5))', 'P06:Clip 0.8-1.0 (m*smooth(0.6,5))', 'P06:DD k=1 (m*smooth(0.6,1), clip0.3-1.5)', 'P06:DD k=5 (m*smooth(0.6,5), clip0.3-1.5)', 'P06:Base (no portfolio overlay)']


## Null hypothesis

Family null (composite): `H0: max_k E[d_{k,t}] ≤ 0`  vs  `H1: ∃ k with E[d_{k,t}] > 0`. Under H0 each column is recentred to zero mean (`d~_{k,t} = d_{k,t} − dbar_k`), the joint matrix is bootstrapped with **shared** stationary-bootstrap indices (cross-strategy dependence preserved — never resampled independently), and at each replication the **maximum** `sqrt(T)·mean` across strategies is recorded. The observed maximum is compared to that null distribution with an **add-one** p-value `p=(k+1)/(B+1)`.

## White's Reality Check


In [3]:
res = {}
for bl in WRC_BLOCKS:
    res[bl] = reality_check(perf, n_resamples=WRC_N, expected_block_length=float(bl), seed=WRC_SEED)
canon = res[WRC_CANON]

wrc_table = pd.DataFrame([{"block length": bl, "best strategy": r.best_strategy,
                           "observed stat": r.best_statistic, "WRC p-value": r.p_value,
                           "null max q95": r.null_q95, "null max q99": r.null_q99}
                          for bl, r in res.items()]).set_index("block length")
print(f"best strategy (highest mean): {canon.best_strategy}")
print(f"observed statistic sqrt(T)*mean: {canon.best_statistic:.4f}")
print(f"canonical (block=20) WRC p-value: {canon.p_value:.4f}  [reject H0 at 0.05: {canon.p_value < 0.05}]")
wrc_table.round(4)

best strategy (highest mean): P04:ls_20pct
observed statistic sqrt(T)*mean: 0.0776
canonical (block=20) WRC p-value: 0.0103  [reject H0 at 0.05: True]


,best strategy,observed stat,WRC p-value,null max q95,null max q99
block length,,,,,
5,P04:ls_20pct,0.0776,0.0038,0.0482,0.0673
10,P04:ls_20pct,0.0776,0.0069,0.0525,0.0719
20,P04:ls_20pct,0.0776,0.0103,0.0559,0.0778
30,P04:ls_20pct,0.0776,0.0119,0.0571,0.0795
50,P04:ls_20pct,0.0776,0.0161,0.0592,0.0836


### Candidate position vs the WRC null-max distribution (diagnostic)

Where does each *current* candidate sit relative to the distribution of the **best** result expected under the no-edge null? Using the **already-generated** canonical (block=20) bootstrap null maxima $M_1^*,\dots,M_B^*$ (reproduced deterministically from the same seed and procedure as the canonical test), we place four strategies on the same $\sqrt{T}\,\bar d$ statistic and the same **gross**-return basis: the WRC family winner `P04:ls_20pct`, plus V1 (incumbent), DD Only, and Combined DD+Vol.

For each we report the observed statistic, its percentile rank within the null-max distribution, the upper-tail exceedance fraction $\hat p_{\text{tail}} = \dfrac{1+\sum_{b=1}^{B} \mathbf 1(M_b^* \ge \bar d_{\text{cand}})}{B+1}$, and whether it clears the 95th / 99th-percentile null-max critical values.

**Interpretation — important.** Only the **family winner's** tail probability corresponds to the canonical WRC test statistic (and equals the family WRC p-value). For V1, DD Only and Combined these tail positions are **descriptive diagnostics, not separate standard WRC p-values** — they are *not* candidate-specific WRC p-values. They simply show where each deployed/candidate strategy falls relative to the distribution of the best-of-family result under the null. The canonical WRC result above is left unchanged.

In [4]:
from src.statistics.bootstrap import stationary_bootstrap_indices

# Reproduce the canonical (block=20) WRC null-max distribution M_1*..M_B* deterministically:
# identical seed + procedure as reality_check() => identical draws (asserted vs canon percentiles).
D = perf.to_numpy().astype(float)
T, K = D.shape
dbar = D.mean(axis=0)
Dc = D - dbar                                   # recentred under the null (col mean 0)
rng = np.random.default_rng(WRC_SEED)
null_max = np.empty(WRC_N, dtype=float)
for b in range(WRC_N):
    idx = stationary_bootstrap_indices(T, float(WRC_CANON), rng)   # shared across strategies
    null_max[b] = np.max(np.sqrt(T) * Dc[idx].mean(axis=0))

q95, q99 = float(np.percentile(null_max, 95)), float(np.percentile(null_max, 99))
assert np.isclose(q95, canon.null_q95, atol=1e-12) and np.isclose(q99, canon.null_q99, atol=1e-12), \
    "reproduced null-max distribution must match the canonical WRC result"

DIAG = {
    "P04:ls_20pct (WRC family winner)": "P04:ls_20pct",
    "V1 (incumbent)": "P06:Deployment (incumbent, as shipped)",
    "DD Only": "P06:DD Only (floor0.3, k5)",
    "Combined DD+Vol": "P06:Combined DD+Vol (floor0.3, k5, clip0.5-1.3)",
}
B = WRC_N
rows = []
for label, col in DIAG.items():
    stat = float(np.sqrt(T) * perf[col].mean())                    # sqrt(T)*mean, same WRC statistic
    pct_rank = float(np.mean(null_max <= stat) * 100.0)            # percentile rank in null-max dist
    p_tail = (1 + int(np.sum(null_max >= stat))) / (B + 1)         # upper-tail exceedance fraction
    rows.append({"strategy": label, "obs mean-return stat": stat,
                 "pctile rank vs null-max": pct_rank, "upper-tail exceedance": p_tail,
                 "> null q95": bool(stat > q95), "> null q99": bool(stat > q99)})
candidate_position = pd.DataFrame(rows).set_index("strategy")

# Consistency: the family winner's tail fraction IS the canonical WRC p-value.
assert np.isclose(candidate_position.loc["P04:ls_20pct (WRC family winner)", "upper-tail exceedance"],
                  canon.p_value, atol=1e-12)
print(f"null-max critical values (block={WRC_CANON}):  q95 = {q95:.4f}   q99 = {q99:.4f}")
print(f"family-winner upper-tail fraction == canonical WRC p-value: {canon.p_value:.4f}")
candidate_position.round(4)

null-max critical values (block=20):  q95 = 0.0559   q99 = 0.0778
family-winner upper-tail fraction == canonical WRC p-value: 0.0103


,obs mean-return stat,pctile rank vs null-max,upper-tail exceedance,> null q95,> null q99
strategy,,,,,
P04:ls_20pct (WRC family winner),0.0776,98.98,0.0103,True,False
V1 (incumbent),0.0667,97.49,0.0252,True,False
DD Only,0.0708,98.19,0.0182,True,False
Combined DD+Vol,0.0738,98.59,0.0142,True,False


## Hansen's SPA

*Not implemented — `superior_predictive_ability` remains a scaffold (a later, more powerful refinement robust to poor strategies in the family). White's Reality Check is the deliverable here.*

## Compare the two tests

*Deferred until Hansen's SPA is implemented.*

## Results table


In [5]:
summary = pd.Series({
    "strategy_universe_size": int(perf.shape[1]),
    "aligned_obs_T": int(perf.shape[0]),
    "returns_basis": "gross (benchmark = 0)",
    "mean_pairwise_corr": round(float(C[iu].mean()), 4),
    "median_pairwise_corr": round(float(np.median(C[iu])), 4),
    "best_strategy": canon.best_strategy,
    "observed_statistic": round(canon.best_statistic, 4),
    "WRC_p_canonical_block20": round(canon.p_value, 4),
    "WRC_p_block_min": round(min(r.p_value for r in res.values()), 4),
    "WRC_p_block_max": round(max(r.p_value for r in res.values()), 4),
    "null_max_q95_block20": round(canon.null_q95, 4),
    "null_max_q99_block20": round(canon.null_q99, 4),
})
summary.to_frame("White Reality Check")

,White Reality Check
strategy_universe_size,23
aligned_obs_T,2557
returns_basis,gross (benchmark = 0)
mean_pairwise_corr,0.9389
median_pairwise_corr,0.9406
best_strategy,P04:ls_20pct
observed_statistic,0.0776
WRC_p_canonical_block20,0.0103
WRC_p_block_min,0.0038
WRC_p_block_max,0.0161


## Evidence ledger

Record every conclusion produced above as an `EvidenceRecord` (`src/statistics/evidence.py`): strategy version, Project 06 freeze tag, dataset, date range, parameters, seed, method, assumptions, statistic, p-value / interval, decision, limitations, timestamp.


In [6]:
from datetime import datetime, timezone
import dataclasses
from src.statistics.evidence import EvidenceRecord, EvidenceLedger, Decision, StatisticalMethod

record = EvidenceRecord(
    record_id="p07_nb06_white_reality_check_v1",
    strategy_version="best-of-family (White RC) = " + canon.best_strategy,
    project06_freeze_tag="project-06-deployment-validation-frozen",
    experiment_id="project_07_reconstructed_strategy_universe",
    dataset="exp_004 strategy_timeseries + exp_006 overlay_equity_curves (23 series, gross)",
    date_range=("2016-01-05", "2026-03-06"),
    parameters={"test": "white_reality_check", "statistic": "sqrt(T)*mean(d_k)", "benchmark": 0.0,
                "n_strategies": int(perf.shape[1]), "n_resamples": WRC_N,
                "block_canonical": WRC_CANON, "blocks_tested": WRC_BLOCKS,
                "p_value_estimator": "add-one (k+1)/(B+1)", "returns_basis": "gross"},
    random_seed=WRC_SEED,
    notebook="research/project_07_statistical_integrity/notebooks/06_white_reality_check.ipynb",
    source_module="src/statistics/white_reality_check.py",
    method=StatisticalMethod.WHITE_REALITY_CHECK, method_version="stat_v1",
    assumptions="Strategy-vs-zero mean-return differential; recentred composite null; shared "
                "stationary-bootstrap indices (cross-strategy dependence preserved); GROSS returns "
                "across a highly-correlated near-duplicate universe (mean corr ~0.94).",
    hypothesis="H0: max_k E[d_k] <= 0 (best searched strategy no better than benchmark after selection).",
    test_statistic=float(canon.best_statistic),
    p_value=float(canon.p_value),
    white_reality_check_result=float(canon.p_value),
    n_trials=int(perf.shape[1]),
    decision=(Decision.SIGNIFICANT if canon.p_value < 0.05 else Decision.NOT_SIGNIFICANT),
    limitations="Conditional on the reconstructed universe and block scheme; missing historical trials "
                "remain a limitation; gross (not net); not a deployment test; best-by-mean-return differs "
                "from the risk-adjusted deployment candidates.",
    timestamp=datetime.now(timezone.utc),
    extra={"block_sensitivity": {str(bl): float(r.p_value) for bl, r in res.items()},
           "null_max_q95_block20": float(canon.null_q95), "null_max_q99_block20": float(canon.null_q99),
           "mean_pairwise_corr": round(float(C[iu].mean()), 4),
           "status": "white reality check completed"})

ledger = EvidenceLedger()
try:
    ledger.append(record); persisted = True
except NotImplementedError:
    persisted = False
OUT = PROJECT_ROOT / "experiments/completed/exp_007_candidate_net_validation"
def _ser(r):
    d = dataclasses.asdict(r); d["method"] = r.method.value; d["decision"] = r.decision.value
    d["timestamp"] = r.timestamp.isoformat(); return d
(OUT / "white_reality_check_record.json").write_text(json.dumps(_ser(record), indent=2))
print("ledger backend persisted:", persisted, "| record saved to exp_007/white_reality_check_record.json")

ledger backend persisted: False | record saved to exp_007/white_reality_check_record.json


## Conclusions

- **Universe:** 23 series-backed strategies (12 P04 base + 11 P06 overlays), 2557 aligned **gross** daily obs, benchmark = 0; mean pairwise correlation ≈ 0.94 (a highly correlated near-duplicate family).
- **Best-of-family (by mean return):** `P04:ls_20pct`, observed `sqrt(T)·mean ≈ 0.078`.
- **White Reality Check:** canonical (block 20) p ≈ 0.010; across blocks [5,10,20,30,50] p ≈ 0.004–0.016 — **rejects H0 at every block length**. The best observed performance is difficult to explain purely as data-snooping across this searched universe.

**Consistency with DSR.** Both point the same way (the edge survives selection), but they answer **related-but-different** questions: DSR asks whether *one selected strategy's Sharpe* survives its trials-inflated benchmark; White's RC asks whether the *maximum mean-return across the whole family* survives the joint selection null. Note the RC winner here (raw `ls_20pct`, highest mean return) is **not** the risk-adjusted deployment candidates (V1/DD Only/Combined), which were selected on Sharpe/drawdown — a different objective.

**Interpretation & caution.** White's RC adjusts for choosing the best of a searched family; it is **not** a deployment test. Failure to reject would not prove all strategies useless; rejection means the best observed strategy is hard to explain as pure data-snooping *under this RC null*. Results remain **conditional on the reconstructed universe** and the block scheme — missing historical trials are a limitation. **No final strategy is selected.**